<a href="https://colab.research.google.com/github/Adidtiasmara/pembelajaran-mesin/blob/main/JS03/JS03-Tugas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
from google.colab import files
uploaded = files.upload()

In [40]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
import numpy as np
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer

# Membaca dataset
df = pd.read_csv("/content/wbc.csv")

# Melihat informasi awal

df.head()





,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [41]:

# Menghapus kolom yang tidak digunakan
kolom_dihapus = ["id", "Unnamed: 32"]
df = df.drop(columns=kolom_dihapus, errors="ignore")

# Encoding kolom diagnosis
df["diagnosis"] = df["diagnosis"].map({
    "M": 1,
    "B": 0
})

# Memisahkan fitur dan target
X = df.drop(columns=["diagnosis"])
y = df["diagnosis"]
y.head()

,diagnosis
0,1
1,1
2,1
3,1
4,1


In [42]:
num_cols = X.columns.tolist()

print("Jumlah fitur:", len(num_cols))
print(num_cols)

Jumlah fitur: 30
['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean', 'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se', 'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst']


In [43]:
# Proses standardisasi data numerik
num_tf = Pipeline([
    ("scaler", StandardScaler())
])

# Preprocessing
preprocess = ColumnTransformer([
    ("num", num_tf, num_cols)
])

In [44]:
# Membagi data menjadi data latih dan data uji
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    stratify=y,
    test_size=0.2,
    random_state=42
)

In [45]:
from sklearn.feature_selection import f_classif

# Seleksi 5 fitur terbaik
selector_filter = SelectKBest(
    score_func=f_classif,
    k=5
)

# Pipeline
pipe_filter = Pipeline([
    ("prep", preprocess),
    ("sel", selector_filter),
    ("clf", LogisticRegression(max_iter=1000))
])

In [46]:
pipe_filter.fit(X_train, y_train)

# Prediksi data uji
pred = pipe_filter.predict(X_test)

print("=== Filter ANOVA + Logistic Regression ===")
print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

=== Filter ANOVA + Logistic Regression ===
Accuracy: 0.9649122807017544
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114



In [54]:
feat_names = pipe_filter.named_steps["prep"].get_feature_names_out()
sel = pipe_filter.named_steps["sel"]

selected_names = feat_names[sel.get_support()]

print("Fitur terbaik:")
print([fitur.replace("num__", "").replace("cat__", "")
       for fitur in selected_names])
pred = pipe_filter.predict(X_test)

print("Jumlah fitur:", len(selected_names))
print("Akurasi:", round(accuracy_score(y_test, pred), 2))

Fitur terbaik:
['radius_mean', 'perimeter_mean', 'area_mean', 'compactness_mean', 'concavity_mean', 'concave points_mean', 'radius_se', 'perimeter_se', 'radius_worst', 'perimeter_worst', 'area_worst', 'compactness_worst', 'concavity_worst', 'concave points_worst']
Jumlah fitur: 14
Akurasi: 0.98


In [55]:
print("Nama fitur:", feat_names)
print("Top fitur:", top)
print(classification_report(y_test, pred))

Nama fitur: ['num__radius_mean' 'num__texture_mean' 'num__perimeter_mean'
 'num__area_mean' 'num__smoothness_mean' 'num__compactness_mean'
 'num__concavity_mean' 'num__concave points_mean' 'num__symmetry_mean'
 'num__fractal_dimension_mean' 'num__radius_se' 'num__texture_se'
 'num__perimeter_se' 'num__area_se' 'num__smoothness_se'
 'num__compactness_se' 'num__concavity_se' 'num__concave points_se'
 'num__symmetry_se' 'num__fractal_dimension_se' 'num__radius_worst'
 'num__texture_worst' 'num__perimeter_worst' 'num__area_worst'
 'num__smoothness_worst' 'num__compactness_worst' 'num__concavity_worst'
 'num__concave points_worst' 'num__symmetry_worst'
 'num__fractal_dimension_worst']
Top fitur: [('num__concave points_worst', np.float64(733.7249325739482)), ('num__perimeter_worst', np.float64(717.2464871041376)), ('num__radius_worst', np.float64(692.861395260564)), ('num__concave points_mean', np.float64(684.5268452011383)), ('num__perimeter_mean', np.float64(548.4132358502825))]
          

---
Berdasarkan hasil pengujian menggunakan SelectKBest dan Logistic Regression, jumlah fitur terbaik yang dapat digunakan adalah 14 fitur. Akurasi yang diperoleh adalah sekitar 98,25%. Fitur-fitur tersebut adalah radius_mean, perimeter_mean, area_mean, compactness_mean, concavity_mean, concave points_mean, radius_se, perimeter_se, radius_worst, perimeter_worst, area_worst, compactness_worst, concavity_worst, dan concave points_worst